# 🌊 Seismic Fault & Horizon Segmentation

**One-click pipeline** for training a U-Net model on the Netherlands F3 Block dataset.

---
**What this notebook does:**
1. Mounts Google Drive & installs dependencies
2. Extracts seismic data + rasterizes geological labels
3. Trains a 9-class U-Net with FP16 mixed precision
4. Evaluates on test set & generates visual overlays

**Runtime:** Select `Runtime` → `Change runtime type` → **T4 GPU**

## 1. Setup & Mount Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# CONFIGURE THIS PATH TO MATCH YOUR DRIVE STRUCTURE
# ============================================================
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/GEO_PROJECT'

# SEG-Y lives inside F3_Demo_2023 (no duplicate needed)
SEGY_PATH = f'{DRIVE_PROJECT_DIR}/F3_Demo_2023/Rawdata/Seismic_data.sgy'
F3_DEMO_DIR = f'{DRIVE_PROJECT_DIR}/F3_Demo_2023'

# Working directories (on Colab's fast local disk)
WORK_DIR = '/content/seismic_project'
DATA_DIR = f'{WORK_DIR}/data'
CHECKPOINT_DIR = f'{WORK_DIR}/checkpoints'
OUTPUT_DIR = f'{WORK_DIR}/outputs'

import os
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify data exists
assert os.path.exists(SEGY_PATH), f'SEG-Y not found at {SEGY_PATH}'
assert os.path.exists(F3_DEMO_DIR), f'F3_Demo_2023 not found at {F3_DEMO_DIR}'
print('✅ Drive mounted and data verified!')
print(f'   SEG-Y: {os.path.getsize(SEGY_PATH) / 1e6:.0f} MB')

In [ ]:
# Install dependencies (only segyio is needed — PyTorch is pre-installed)
!pip install -q segyio

# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Copy source files from Drive to Colab local disk (faster I/O)
SRC_DRIVE = f'{DRIVE_PROJECT_DIR}/src'
SRC_LOCAL = f'{WORK_DIR}/src'

!cp -r {SRC_DRIVE} {SRC_LOCAL}

# Add project to Python path
import sys
sys.path.insert(0, WORK_DIR)

print('✅ Source files copied and path configured')
!ls -la {SRC_LOCAL}/

## 2. Data Pipeline — Extract Seismic Slices & Labels

This step:
- Loads the 699MB SEG-Y file
- Parses 7 horizon + 1 fault interpretation files
- Rasterizes them into pixel-level masks
- Saves paired .npy files (amplitude + mask) for each inline

**⏱ Expected time: ~10–15 minutes**

In [ ]:
from src.data_pipeline import run_pipeline

run_pipeline(
    segy_path=SEGY_PATH,
    f3_demo_dir=F3_DEMO_DIR,
    output_dir=DATA_DIR,
)

In [ ]:
# Quick sanity check — visualize one extracted slice + mask
import numpy as np
import matplotlib.pyplot as plt

# Load a sample
slice_files = sorted(os.listdir(f'{DATA_DIR}/slices'))[:1]
sample_name = slice_files[0]
amp = np.load(f'{DATA_DIR}/slices/{sample_name}')
mask = np.load(f'{DATA_DIR}/masks/{sample_name.replace(".npy", "_mask.npy")}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.imshow(amp, cmap='gray', aspect='auto')
ax1.set_title(f'Amplitude: {sample_name}')
ax2.imshow(mask, cmap='tab10', aspect='auto', vmin=0, vmax=8)
ax2.set_title('Label Mask')
plt.tight_layout()
plt.show()

# Print class distribution for this slice
unique, counts = np.unique(mask, return_counts=True)
CLASS_NAMES = ['Background', 'Fault', 'FS4', 'MFS4', 'FS6', 'FS7', 'FS8', 'Shallow', 'Top Foresets']
for u, c in zip(unique, counts):
    print(f'  Class {u} ({CLASS_NAMES[u]:>14s}): {c:>8,} pixels')

## 3. Training

Trains U-Net with:
- **Composite Loss** (Weighted CE + Dice)
- **FP16 Mixed Precision** for memory efficiency
- **Cosine Annealing LR** scheduler
- Best model saved by validation Dice score

**⏱ Expected time:**
- 5 epochs (MVP test): ~5 min
- 50 epochs (good results): ~45 min
- 80 epochs (best results): ~1.5 hrs

In [ ]:
# ============================================================
# TRAINING CONFIGURATION — Adjust as needed
# ============================================================
EPOCHS = 50            # Start with 5 for quick test, then 50-80
BATCH_SIZE = 16        # T4 can handle 16 easily with FP16
LEARNING_RATE = 1e-4   # Adam default, works well for U-Net

from src.train import train

best_model_path = train(
    data_dir=DATA_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    num_workers=2,
)

In [ ]:
# Plot training curves
checkpoint = torch.load(best_model_path, map_location='cpu')
history = checkpoint['history']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(history['train_loss'], label='Train Loss', color='#2196F3', linewidth=2)
ax1.plot(history['val_loss'], label='Val Loss', color='#F44336', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Dice curve
ax2.plot(history['val_dice'], label='Val Dice', color='#4CAF50', linewidth=2)
ax2.axhline(y=max(history['val_dice']), color='gray', linestyle='--', alpha=0.5,
            label=f'Best: {max(history["val_dice"]):.4f}')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice Score')
ax2.set_title('Validation Dice Score')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best Dice: {max(history["val_dice"]):.4f} at epoch {np.argmax(history["val_dice"]) + 1}')

## 4. Evaluation

Runs inference on the held-out test set and generates:
- Per-class IoU, Dice, Precision, Recall metrics
- Visual overlay images (seismic + predicted faults/horizons)

In [ ]:
from src.evaluate import evaluate

evaluate(
    data_dir=DATA_DIR,
    checkpoint_path=f'{CHECKPOINT_DIR}/best_model.pth',
    output_dir=OUTPUT_DIR,
    max_overlays=10,
)

In [ ]:
# Display the generated overlay images
from IPython.display import display, Image as IPImage
import glob

overlay_files = sorted(glob.glob(f'{OUTPUT_DIR}/*_pred.png'))[:6]

for f in overlay_files:
    print(f'\n{os.path.basename(f)}')
    display(IPImage(filename=f, width=900))

## 5. Save Results to Drive

Copy the trained model and outputs back to Google Drive for persistence.

In [ ]:
# Copy results back to Drive
DRIVE_OUTPUT = f'{DRIVE_PROJECT_DIR}/results'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

!cp -r {CHECKPOINT_DIR} {DRIVE_OUTPUT}/
!cp -r {OUTPUT_DIR} {DRIVE_OUTPUT}/

print(f'\n✅ Results saved to Google Drive: {DRIVE_OUTPUT}/')
print(f'   Checkpoints: {DRIVE_OUTPUT}/checkpoints/')
print(f'   Overlays:    {DRIVE_OUTPUT}/outputs/')